## Uses more lenient constraints so should have more stars be accepted

In [1]:
from astropy.io import fits
import numpy as np

test_stars = ["PIC 3055651000014","PIC 3019516000004","PIC 3016509000091","PIC 3006117000071","PIC 2993118000281","PIC 2990953000149","PIC 2953270000139","PIC 2914127000043","PIC 2901793000033","PIC 2893299000027","PIC 2878835000060","PIC 2864161000018","PIC 2857990000007","PIC 2850548000009","PIC 2841226000121","PIC 2834928000134","PIC 2807525000024","PIC 2805893000144","PIC 2800939000054","PIC 2797563000103","PIC 2795832000048","PIC 2787400000022","PIC 2778992000018","PIC 2773872000124","PIC 2772178000055","PIC 2768785000027","PIC 2768655000123","PIC 2766761000048","PIC 2761564000023","PIC 2754583000062","PIC 2752733000008","PIC 2749182000061","PIC 2736759000143","PIC 2736701000108","PIC 2699647000039","PIC 2682526000062","PIC 2588238000213","PIC 2584223000150","PIC 2577865000019","PIC 2563636000091","PIC 2459261000546","PIC 2436634000119","PIC 2416198000076","PIC 2395685000232","PIC 2383361000076","PIC 2299424000182","PIC 2256289000040","PIC 2207252000011","PIC 2881985000315","PIC 2426236000058"]
fits_file = r"C:/Users/jshld/Downloads/LOPS2PICtarget2.1.0.1-t-fg-c-scv.fits"
#rename to your stuff


def detectable_lenient(names, fits_file):
    #input the list of stars and then threshold is applied, might take a little while
    
    sun_nu_max = 3090
    sun_Teff = 5770

    def threshold_function(x):
        a = 1.9619
        b = -10.1070
        c = 0.00000317
        x0 = 1830.07
        return 10 ** ((-a + c * (x - x0)**2) / b)

    names = np.atleast_1d(names)

    with fits.open(fits_file) as hdul:
        data = hdul[1].data

        mask = np.isin(data['PICname'], names)
        star_data = data[mask]

        Teff = star_data['Teff']
        Radius = star_data['Radius']
        Mass = star_data['Mass']
        sig1hr = star_data['BOLrandomSysNSRNCAM_T']
        picnames = star_data['PICname']

    nu_max = sun_nu_max * (Mass / Radius**2) * (Teff / sun_Teff) ** (-0.5)
    background = 2 * (sig1hr**2) * 3600 * (10**(-6))
    threshold = threshold_function(nu_max)

    is_detectable = background <= threshold

    return is_detectable

In [5]:
acceptance_mask = detectable_lenient(test_stars,fits_file)

In [6]:
print(np.array(test_stars)[acceptance_mask])

['PIC 3006117000071' 'PIC 2914127000043' 'PIC 2787400000022'
 'PIC 2768785000027' 'PIC 2761564000023' 'PIC 2754583000062'
 'PIC 2752733000008' 'PIC 2577865000019' 'PIC 2459261000546']
